<a href="https://colab.research.google.com/github/internshipinabook/data-science-internshipinabook/blob/main/notebooks/week6_feature_engineering_STARTER.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

> ⏸️ **Pause and Predict**
> Before engineering any feature: list three domain-driven features you could create from the existing columns. For each, write one sentence explaining why a data scientist at a telecom company would expect it to predict churn.

In [ ]:
# ── Colab setup (skip if running locally) ────────────────────────────────────
import os
if not os.path.exists('requirements.txt'):
    !git clone https://github.com/internshipinabook/data-science-internshipinabook.git book
    %cd book
    !pip install -r requirements.txt -q
    print('Colab setup complete ✅')
else:
    print('Running locally ✅')


# Week 6 — Feature Engineering
### Week 6 — The Data Science Internship | Kova Analytics

> **STARTER notebook.**

The Data Science Internship Book 0 of 9 in the InternshipInABook™ Series

This repository contains the practical exercises from the book.

📖 Complete internship experience:

Workplace scenarios
Mentor guidance
Reflection exercises
Portfolio
checkpoints
Capstone projects
👉 Get the complete book: https://selar.com/al990ay7ux

🚀 Start Here
Welcome to The Data Science Internship.

If you are new to the InternshipInABook™ Series:

Run this setup notebook.
Complete Week 1.
Follow the weekly internship schedule.
Build your portfolio.
Complete the capstone project.
🔗 Repository: https://github.com/internshipinabook/data-science-internshipinabook

Run every cell top to bottom. Each cell prints ✅ if everything is working or ❌ with a fix instruction. Complete every fix before moving to Week 1.

---

## ⚠️ Common Mistakes

| Mistake | What goes wrong | Fix |
|---------|-----------------|-----|
| Engineering features before splitting train/test | Any feature computed on the full dataset leaks future information into training. Split first, engineer after. | Always apply feature engineering to train set only, then transform test set |
| Creating features without checking their importance | Not every engineered feature helps. Check SHAP or feature importance after adding new columns. | Run feature importance after every new feature batch and drop anything below threshold |

## 🏆 Challenge Task

> 🎯 **Core Path**
> Create two new features. For each: write the business rationale, implement it, and check whether it improves model F1 vs the Week 5 baseline.

> 🔬 **Advanced Path**
> Implement a full sklearn Pipeline that combines preprocessing, feature engineering, and model fitting in one `.fit()` call. Run it end-to-end from raw data.

## ✅ What You Can Do After This Week

- Engineer at least four features from raw columns with documented hypotheses
- Apply one-hot encoding with column sanitisation and feature alignment
- Compare logistic regression and random forest on equal footing
- Extract and interpret feature importance with three named caveats
- Translate a model's top feature into a plain-language business insight

*Add `week6_feature_engineering.ipynb` to your internship portfolio.*

<a href="https://colab.research.google.com/github/internshipinabook/data-science-internshipinabook/blob/main/notebooks/week6_feature_engineering_STARTER.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>&nbsp;&nbsp;
<a href="https://github.com/internshipinabook/data-science-internshipinabook/blob/main/notebooks/week6/week6_feature_engineering_STARTER.ipynb">
  <img src="https://img.shields.io/badge/View%20on-GitHub-181717?logo=github" alt="View on GitHub"/>
</a>

---

## 🖥️ How to Run This Notebook

| Platform | Instructions |
|----------|-------------|
| **Google Colab** | Click the badge above — free, no setup needed |
| **Local Jupyter** | `pip install -r requirements.txt` then `jupyter lab` |
| **VS Code** | Open `.ipynb` with the Jupyter extension installed |
| **GitHub** | Click "View on GitHub" above for a read-only preview |

> **STARTER notebook** — Complete the `# YOUR CODE HERE` cells. Check the SOLUTION notebook if stuck.

In [ ]:
# ── PLATFORM SETUP ───────────────────────────────────────────────────────────
# Run this cell first. It installs missing libraries in Google Colab.
# In a local environment, skip it if requirements.txt is already installed.

import sys, subprocess

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print("📦 Google Colab detected — installing libraries...")
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    'pandas>=1.5', 'numpy>=1.23', 'matplotlib>=3.6',
                    'seaborn>=0.12', 'scikit-learn>=1.2'], check=True)
    print("✅ Libraries installed.")
else:
    print("💻 Running locally.")
    print("   If you see ImportError below, run: pip install -r requirements.txt")

## 📂 Dataset

The dataset is included in the course repository. No Kaggle account required.

**File:** `customer_churn.csv`  
**Source:** `https://raw.githubusercontent.com/internshipinabook/data-science-internshipinabook/main/data/customer_churn.csv`

Run the cell below to load it directly.

## 🔄 Adaptability — Use Your Own Dataset

This notebook is written for the Telco Churn dataset but the **entire ML pipeline works on any binary classification problem**.

**To swap in your own data, change only these lines at the top of the notebook:**
```python
DATA_FILE      = 'customer_churn.csv'  # ← your CSV filename
TARGET_COL     = 'Churn'              # ← the binary column you want to predict (Yes/No or 1/0)
POSITIVE_LABEL = 'Yes'                # ← which value means "positive" (the event you care about)
ID_COL         = 'customerID'         # ← identifier column to drop before modelling (or None)
NUMERIC_FIX    = 'TotalCharges'       # ← any numeric column stored as text (or None)
```

**Works with any binary classification dataset:**
- 🏦 Loan default prediction
- 🏥 Patient readmission risk
- 📧 Email spam detection
- 💳 Credit card fraud
- 🛒 Customer conversion
- 🎓 Student dropout prediction

> ⚠️ **Class imbalance warning:** If your dataset has <20% positive class, use `class_weight='balanced'` — already done in this notebook. Check your class distribution in Step 4 before proceeding.

---

## 🔄 Using a Different Dataset?

This notebook is written for `customer_churn.csv` but the ML pipeline applies to any binary classification problem.

**To adapt:**
1. Change `DATA_FILE` to your filename
2. Set `TARGET_COL` to your binary target column (the thing you want to predict)
3. Update `POSITIVE_LABEL` to the positive class name (e.g. `'Yes'`, `'1'`, `'Fraud'`)
4. The `prepare_dataset()` function encodes features — update the column references to match yours

```python
# ── Adapt these for your dataset ─────────────────────────────────────────────
DATA_FILE      = 'customer_churn.csv'   # ← change to your file
TARGET_COL     = 'Churn'               # ← your binary target column
POSITIVE_LABEL = 'Yes'                 # ← the positive class value
ID_COL         = 'customerID'          # ← identifier column to drop (or None)
```

**Works with any binary classification dataset** — fraud detection, disease prediction,
loan default, email spam, customer conversion — as long as your target has two classes.

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold, GridSearchCV, cross_validate
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (recall_score, precision_score, f1_score,
                              classification_report, confusion_matrix,
                              ConfusionMatrixDisplay, roc_curve, auc,
                              precision_recall_curve, accuracy_score)
%matplotlib inline
sns.set_theme(style='whitegrid')
print('✅ Libraries loaded')

In [ ]:
SERVICE_COLS = ['PhoneService','MultipleLines','InternetService','OnlineSecurity',
                'OnlineBackup','DeviceProtection','TechSupport','StreamingTV','StreamingMovies']

def prepare_dataset(df):
    df = df.copy()
    df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
    df['TotalCharges'] = df['TotalCharges'].fillna(0)  # new customers have zero charges
    if 'customerID' in df.columns:
        df = df.drop(columns=['customerID'])
    le = LabelEncoder()
    df['Churn'] = le.fit_transform(df['Churn'].astype(str))
    df['tenure_charge_ratio'] = df['TotalCharges'] / (df['tenure'] + 1)
    sc = [c for c in SERVICE_COLS if c in df.columns]
    for col in sc:
        df[f'{col}_binary'] = df[col].apply(lambda x: 1 if str(x).lower()=='yes' else 0)
    df['num_services'] = df[[f'{c}_binary' for c in sc]].sum(axis=1)
    contract_map = {'Month-to-month':2,'One year':1,'Two year':0}
    df['contract_numeric'] = df['Contract'].map(contract_map)
    df['paperless_numeric'] = (df['PaperlessBilling']=='Yes').astype(int)
    df['contract_paperless_interaction'] = df['contract_numeric'] * df['paperless_numeric']
    df['tenure_segment'] = pd.cut(df['tenure'], bins=[0,12,36,float('inf')],
                                   labels=['New','Established','Loyal'])
    return df
print('✅ prepare_dataset() defined')

## Hypotheses Table

| Feature | Calculation | Hypothesis | Expected Effect |
|---------|-------------|------------|-----------------|
| `tenure_charge_ratio` | | | |
| `num_services` | | | |
| `contract_paperless_interaction` | | | |
| `tenure_segment` | | | |

## 1. Load and apply prepare_dataset()

In [ ]:
# YOUR CODE HERE


## 2. Feature 1 — Tenure Charge Ratio: distribution chart (churned vs not)

In [ ]:
# YOUR CODE HERE


## 3. Feature 2 — Number of Services: churn rate chart with sample sizes

In [ ]:
# YOUR CODE HERE


## 4. Feature 3 — Contract×Billing Interaction: pivot table

In [ ]:
# YOUR CODE HERE


## 5. Feature 4 — Tenure Segment (pd.cut): churn rate chart

In [ ]:
# YOUR CODE HERE


## 6. Low-signal feature: find one with evidence

In [ ]:
# YOUR CODE HERE


## 7. Build full feature set, split, scale

In [ ]:
# YOUR CODE HERE


## 8. LR and RF — both with class_weight='balanced'

In [ ]:
# YOUR CODE HERE


## 9. Side-by-side comparison (text reports + visual comparison chart)

In [ ]:
# YOUR CODE HERE


## 10. Feature Importance — top 15, horizontal bar chart

In [ ]:
# YOUR CODE HERE


## 11. Five-Section Model Report
### 1 — What Changed From Week 5
### 2 — Model Comparison
### 3 — Feature Importance
### 4 — Business Interpretation
### 5 — Open Questions

---
## ✅ Week 6 Complete
**Next:** `week7/*_STARTER.ipynb`

---
*github.com/InternshipInABook*